# Duration-PIT × architecture × propagation factorial

Lean research gate for the current scenario-engine architecture.

The factorial compares:
- representation: raw target-duration empirical law vs duration-conditioned PIT pooling;
- architecture: blockwise vs hierarchical global/local dependence;
- propagation: direct-H vs recursive-1W;
- scores: multivariate Energy Score and dependence-sensitive Variogram Score.

This notebook is deliberately separate from the main engine so model adjudication does not make the
standalone repricing workflow slower or harder to read.


In [1]:
# Lean imports + configuration (only what this gate uses)
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web
from scipy.optimize import minimize
from scipy.stats import norm

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

# Compact cross-asset observable panel
TICKERS = ["MSFT", "JPM", "XOM", "JNJ", "PG", "HD", "CAT", "NEM", "PLD", "DUK"]
MARKET_PROXY = "SPY"                  # downloaded by the legacy helper; not used in the gate
RATE_TICKERS = ["DGS2", "DGS10", "DGS30"]
FX_SERIES = {"DEXUSEU": "EURUSD"}
MOODYS_SERIES = {"DAAA": "Aaa", "DBAA": "Baa"}

AS_OF_DATE = pd.Timestamp("2026-05-01")
DOWNLOAD_END = (AS_OF_DATE + pd.Timedelta(days=10)).strftime("%Y-%m-%d")
EQUITY_START = "1995-01-01"
FX_START = "1999-01-04"
MOODYS_START_DATE = "1986-01-01"

HORIZON_BDAYS_BY_MODE = {"1W": 5, "1M": 21, "3M": 63, "1Y": 252}

# Random-duration PIT settings inherited from v27/v29
V27_PIT_CLIP = 1e-4
V27_PW_LOOKBACK_DAYS = 1260
V27_BLOCK_MAX_DAYS = 126
BOOTSTRAP_EXPECTED_BLOCK = 10

# v30 research-gate settings: deliberately lean, not a publication notebook
V30_SCENARIOS = 256
V30_BLOCK_LIBRARY_SIZE = 700
V30_HIER_MAX_GLOBAL = 3
V30_BASE_HORIZON_DAYS = 5
V30_MIN_TRAIN_DAYS = 750
V30_MAX_ORIGINS_BY_HORIZON = {"1W": 200, "1M": 200, "3M": 80, "1Y": 25}
V30_BOOTSTRAP_REPS = 999
V30_VARIOGRAM_P = 0.5
V30_SEED = 30030


In [2]:
def download_market_data(tickers, market_proxy, rate_tickers, start_date, download_end):
    """Download adjusted-close equity prices, a market proxy, and FRED Treasury key rates."""
    all_equity_tickers = list(dict.fromkeys(list(tickers) + [market_proxy]))
    stock_raw = yf.download(
        all_equity_tickers,
        start=start_date,
        end=download_end,
        auto_adjust=False,
        progress=False,
        actions=False,
    )

    if isinstance(stock_raw.columns, pd.MultiIndex):
        if "Adj Close" not in stock_raw.columns.get_level_values(0):
            raise ValueError("Adjusted close prices not found in yfinance download output.")
        prices_all = stock_raw["Adj Close"].copy()
    else:
        prices_all = stock_raw.copy()
        if isinstance(prices_all, pd.Series):
            prices_all = prices_all.to_frame(name=all_equity_tickers[0])
        prices_all.columns = all_equity_tickers

    stock_prices = prices_all[tickers].copy()
    market_proxy_prices = prices_all[[market_proxy]].copy()
    rates_raw = web.DataReader(rate_tickers, "fred", start_date, download_end).astype(float) / 100.0
    return stock_prices, market_proxy_prices, rates_raw


def download_fx_rates(fx_series, start_date, download_end):
    """Download FX spot rates from FRED (one column per currency pair).

    `fx_series` is an ordered mapping {FRED_code: pair_label}. FRED quotes these pairs as
    'US dollars per 1 unit of foreign currency' (e.g. DEXUSEU = USD per EUR), so each column is the
    price in USD of one unit of the foreign currency and a long-foreign position gains when the
    column rises. Unlike the ICE BofA credit series these spot series carry long history with no
    3-year truncation.
    """
    codes = list(fx_series.keys())
    fx = web.DataReader(codes, "fred", start_date, download_end).astype(float)
    fx = fx.rename(columns=dict(fx_series))
    fx = fx[[fx_series[c] for c in codes]]
    return fx


def download_moodys_credit(moodys_series, start_date, download_end):
    """Download Moody's seasoned corporate bond yields from FRED (long history, no 3-year truncation).

    `moodys_series` maps FRED codes to labels, e.g. {"DAAA": "Aaa", "DBAA": "Baa"}. These daily
    seasoned-yield series run back to the 1980s and carry every postwar credit crisis, so the
    Baa-Aaa spread is used as a long-history credit-stress backbone to calibrate how often and how
    severely credit blows out relative to the short, calm OAS window the main engine is fit on.
    Yields are in percent on FRED; divided by 100 to decimals here.
    """
    codes = list(moodys_series.keys())
    mood = web.DataReader(codes, "fred", start_date, download_end).astype(float) / 100.0
    mood = mood.rename(columns=dict(moodys_series))
    return mood


def compute_equity_log_returns(prices):
    return np.log(prices).diff().dropna()


def compute_yield_changes(yields):
    return yields.diff().dropna()


def compute_spread_changes(spreads):
    """Daily changes in credit OAS levels (decimals), analogous to yield changes."""
    return spreads.diff().dropna()


def _offblock_lowrank_fit(S, mask_off, labels, k, n_iter=3000, tol=1e-10, max_block_share=0.95):
    """Least-squares rank-k fit L L' to the OFF-block entries of S only (multi-battery factor analysis,
    unweighted least squares), the diagonal blocks treated as missing and imputed by the current fit
    (hard-impute iteration). A Heywood guard keeps the fitted global variance inside each block below
    `max_block_share` of the block's own variance. Used only to seed `_crossblock_factor_weights`."""
    blocks = list(dict.fromkeys(labels.tolist()))
    idx = {b: np.where(labels == b)[0] for b in blocks}
    inv_half = {}
    for b in blocks:
        ev, V = np.linalg.eigh(np.atleast_2d(S[np.ix_(idx[b], idx[b])]))
        ev = np.clip(ev, ev.max() * 1e-12, None)
        inv_half[b] = (V / np.sqrt(ev)) @ V.T
    A = S.copy()
    off_energy = float(np.sum((S * mask_off) ** 2))
    prev, best, it = None, None, 0
    for it in range(1, int(n_iter) + 1):
        ev, V = np.linalg.eigh(A)
        top = np.argsort(ev)[::-1][:k]
        L = V[:, top] * np.sqrt(np.clip(ev[top], 0.0, None))
        for b in blocks:
            Lb = L[idx[b]]
            r = float(np.linalg.eigvalsh(np.atleast_2d(inv_half[b] @ Lb @ Lb.T @ inv_half[b]))[-1])
            if r > max_block_share:
                L[idx[b]] = Lb * np.sqrt(max_block_share / r)
        LL = L @ L.T
        obj = float(np.sum(((S - LL) * mask_off) ** 2))
        if best is None or obj < best[1]:
            best = (L.copy(), obj)
        if prev is not None and abs(prev - obj) <= tol * max(off_energy, 1e-300):
            break
        prev = obj
        A = np.where(mask_off, S, LL)
    return best[0], best[1], off_energy, it


def _bartlett_weights(S, L, labels):
    """Bartlett factor-score weights for loadings L under a block-diagonal uniqueness
    Psi = blockdiag(S - L L'), repaired to positive definite block by block (a starting point only)."""
    Psi = np.zeros_like(S)
    for b in dict.fromkeys(labels.tolist()):
        m = np.where(labels == b)[0]
        B = S[np.ix_(m, m)] - L[m] @ L[m].T
        ev, V = np.linalg.eigh(np.atleast_2d(B))
        floor = 1e-3 * float(np.trace(np.atleast_2d(S[np.ix_(m, m)]))) / len(m)
        Psi[np.ix_(m, m)] = (V * np.clip(ev, floor, None)) @ V.T
    PiL = np.linalg.solve(Psi, L)
    return PiL @ np.linalg.inv(L.T @ PiL)


def _crossblock_objective(x, S, mask_off, kg):
    """Cross-block covariance energy left after regressing the drivers on g = Ds W, with its analytic
    gradient: f(W) = ||offblock(S - S W (W'SW)^-1 W'S)||_F^2 (v24)."""
    W = x.reshape(S.shape[0], kg)
    SW = S @ W
    try:
        Ai = np.linalg.inv(W.T @ SW)
    except np.linalg.LinAlgError:
        return 1e300, np.zeros_like(x)
    G = mask_off * (S - SW @ Ai @ SW.T)
    SGS = S @ G @ S
    grad = -4.0 * (SGS @ W @ Ai - SW @ (Ai @ (W.T @ SGS @ W) @ Ai))
    return float(np.sum(G * G)), grad.ravel()


def _crossblock_factor_weights(S, mask_off, labels, kg, extra_starts=()):
    """Weights W (k, kg) of the kg linear combinations of the block-normalized drivers whose removal by
    regression leaves the LEAST cross-block covariance (v24). Smooth and invariant to W -> W Q; solved by
    L-BFGS with the analytic gradient from several starts (a multi-battery least-squares factor fit's
    Bartlett weights, the leading eigenvectors, and any supplied start)."""
    starts = [np.asarray(s_, dtype=float) for s_ in extra_starts]
    L, _, _, _ = _offblock_lowrank_fit(S, mask_off, labels, kg, n_iter=300)   # a seed: a few hundred iterations suffice
    starts.append(_bartlett_weights(S, L, labels))
    ev, V = np.linalg.eigh(S)
    starts.append(V[:, np.argsort(ev)[::-1][:kg]])
    best = None
    for W0 in starts:
        r = minimize(_crossblock_objective, W0.ravel(), args=(S, mask_off, kg), jac=True,
                     method="L-BFGS-B", options={"maxiter": 5000})
        if best is None or r.fun < best.fun:
            best = r
    return best.x.reshape(S.shape[0], kg)


def _nested_unit_factors(Ds, W, w, prev_g=None):
    """Factors spanning Ds @ W, orthonormal under the weights w (sum w = 1), ordered so the first ones are
    as close as possible to the previous (k-1) solution: g_1 is always the best single global factor, g_2
    the best addition to it, and so on. Returns (T, kg)."""
    g = Ds @ W
    ev, V = np.linalg.eigh(np.atleast_2d((g * w[:, None]).T @ g))
    h = g @ (V / np.sqrt(np.clip(ev, 1e-300, None)))
    kg = h.shape[1]
    if prev_g is None or prev_g.shape[1] == 0:
        return h[:, ::-1]
    A = (h * w[:, None]).T @ prev_g
    Q, _ = np.linalg.qr(np.hstack([A, np.eye(kg)]))
    Q = Q[:, :kg]
    for j in range(A.shape[1]):
        if float(Q[:, j] @ A[:, j]) < 0:
            Q[:, j] *= -1.0
    return h @ Q


def _flat_top_kernel(x):
    """Politis (2003) flat-top lag window: 1 on [0, 1/2], linearly down to 0 on [1/2, 1], 0 beyond."""
    w = np.ones_like(x, dtype=float)
    mid = (x >= 0.5) & (x <= 1.0)
    w[mid] = 2.0 * (1.0 - x[mid])
    w[x > 1.0] = 0.0
    return w


def politis_white_block_length(x):
    """Automatic block-length selection for the stationary bootstrap (Politis & White 2004, with the
    Patton-Politis-White 2009 correction), computed per column.

    Returns the optimal *expected* block length per column. The optimum is
    b = (2 G^2 / D_SB)^{1/3} n^{1/3} with G = sum_k lam(k/M)|k|R(k), D_SB = 2 (sum_k lam(k/M)R(k))^2,
    R the autocovariance and M a flat-top bandwidth chosen where the autocorrelation becomes
    negligible. Because the procedure reads the *level* autocovariance, applying it to returns gives a
    short block (returns are nearly serially uncorrelated) while applying it to the absolute/squared
    series captures volatility clustering -- which is exactly why, after the FHS filter has removed the
    clustering, the standardized residuals admit a short block.
    """
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        x = x[:, None]
    n, k = x.shape
    kn = max(5, int(np.ceil(np.sqrt(np.log10(n)))))
    mmax = int(np.ceil(np.sqrt(n))) + kn
    bmax = float(np.ceil(min(3.0 * np.sqrt(n), n / 3.0)))
    cval = 2.0 * np.sqrt(np.log10(n) / n)
    out = np.empty(k)
    for j in range(k):
        e = x[:, j] - x[:, j].mean()
        acv = np.array([np.sum(e[:n - lag] * e[lag:]) / n for lag in range(mmax + 1)])
        if acv[0] <= 0:
            out[j] = 1.0
            continue
        rho = acv / acv[0]
        sig = np.abs(rho) > cval
        mhat = None
        for mm in range(1, mmax - kn + 1):
            if not sig[mm + 1:mm + 1 + kn].any():
                mhat = mm
                break
        if mhat is None:
            mhat = mmax
        M = int(min(2 * mhat, mmax))
        lags = np.arange(-M, M + 1)
        w = _flat_top_kernel(np.abs(lags) / M)
        g_acv = acv[np.abs(lags)]
        g_hat = np.sum(w * np.abs(lags) * g_acv)
        g0 = np.sum(w * g_acv)
        if g0 <= 0 or g_hat == 0:
            out[j] = 1.0
        else:
            b = (2.0 * g_hat ** 2 / (2.0 * g0 ** 2)) ** (1.0 / 3.0) * n ** (1.0 / 3.0)
            out[j] = float(min(max(b, 1.0), bmax))
    return out


def _v27_observable_panel():
    """Additive daily cross-asset observables for the duration-PIT gate.

    Equity and FX are log returns; rates and credit are daily level changes.  Therefore an L-row sum is
    exactly the 0->L log return / yield change / spread change used by the duration-conditioned ECDF.
    A compact panel is intentional: this gate tests the representation before it is wired into every
    compressed factor of the full repricing engine.
    """
    eq = compute_equity_log_returns(stock_prices_raw[TICKERS]).mean(axis=1).rename("EQ_EW")
    dy = compute_yield_changes(rates_raw)
    rate_pref = [c for c in ("DGS2", "DGS10", "DGS30") if c in dy.columns]
    if len(rate_pref) < 2:
        rate_pref = list(dy.columns[:min(3, len(dy.columns))])
    rp = dy[rate_pref].copy()
    rp.columns = [f"RATE_{c}" for c in rp.columns]

    ds = compute_spread_changes(credit_driver_raw)
    credit_pref = [c for c in ("Aaa_spr", "Baa_spr", "AAA", "BBB", "BB") if c in ds.columns]
    if len(credit_pref) < 2:
        # choose a small, deterministic subset; the gate is about cross-block dependence, not bucket coverage
        credit_pref = list(ds.columns[:min(3, len(ds.columns))])
    else:
        credit_pref = credit_pref[:3]
    cp = ds[credit_pref].copy()
    cp.columns = [f"CREDIT_{c}" for c in cp.columns]

    parts = [eq.to_frame(), rp, cp]
    if 'fx_rates_raw' in globals() and fx_rates_raw is not None and len(fx_rates_raw.columns):
        fx = np.log(fx_rates_raw / fx_rates_raw.shift(1)).copy()
        fx.columns = [f"FX_{c}" for c in fx.columns]
        parts.append(fx)
    P = pd.concat(parts, axis=1).dropna().sort_index()
    labels = np.array([
        "equity" if c.startswith("EQ_") else
        "rates" if c.startswith("RATE_") else
        "credit" if c.startswith("CREDIT_") else
        "fx" for c in P.columns
    ], dtype=object)
    return P, labels


def _v27_precompute_duration_aggregates(panel, max_len):
    """All overlapping L-day additive moves, indexed by block END date, for L=1..max_len."""
    A = panel.to_numpy(dtype=float)
    cs = np.vstack([np.zeros((1, A.shape[1])), np.cumsum(A, axis=0)])
    out = {}
    n = len(panel)
    for L in range(1, min(int(max_len), n - 1) + 1):
        vals = cs[L:] - cs[:-L]
        out[L] = pd.DataFrame(vals, index=panel.index[L-1:], columns=panel.columns)
    return out


def _v27_pit_apply(values, sorted_marginals, clip=1e-4):
    """Training-ECDF PIT -> N(0,1) score, column by column; no future ranks enter the map."""
    X = np.atleast_2d(np.asarray(values, dtype=float))
    S = np.asarray(sorted_marginals, dtype=float)
    n, k = S.shape
    Z = np.empty_like(X)
    for j in range(k):
        r = np.searchsorted(S[:, j], X[:, j], side="right")
        u = (r + 0.5) / (n + 1.0)
        Z[:, j] = norm.ppf(np.clip(u, clip, 1.0 - clip))
    return Z


def _v27_sorted_marginals(agg_train):
    return np.sort(np.asarray(agg_train, dtype=float), axis=0)


def _v27_rank_map_to_duration(z_draws, sorted_target):
    """Keep the simulated copula/ranks, force every marginal back to the target L-day empirical law."""
    Z = np.asarray(z_draws, dtype=float)
    n, k = Z.shape
    U = np.empty_like(Z)
    for j in range(k):
        r = pd.Series(Z[:, j]).rank(method="average").to_numpy(dtype=float)
        U[:, j] = (r - 0.5) / n
    S = np.asarray(sorted_target, dtype=float)
    qgrid = (np.arange(S.shape[0], dtype=float) + 0.5) / S.shape[0]
    X = np.empty_like(Z)
    for j in range(k):
        X[:, j] = np.interp(np.clip(U[:, j], qgrid[0], qgrid[-1]), qgrid, S[:, j])
    return X


def _v27_expected_block_length(train):
    """PW expected length; PR will draw the realised geometric runs.

    The gate operates on raw observables (no GJR layer here), so the automatic selector reads absolute
    standardized moves to retain economically relevant volatility persistence rather than collapsing to
    the near-iid sign/level autocorrelation of daily returns.
    """
    X = train.iloc[-min(len(train), int(V27_PW_LOOKBACK_DAYS)):].copy()
    sd = X.std(ddof=0).replace(0.0, np.nan)
    Z = ((X - X.mean()) / sd).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    bcol = politis_white_block_length(np.abs(Z.to_numpy(dtype=float)))
    b = float(np.nanmedian(bcol[np.isfinite(bcol)])) if np.isfinite(bcol).any() else float(BOOTSTRAP_EXPECTED_BLOCK)
    return float(np.clip(b, 1.0, V27_BLOCK_MAX_DAYS)), bcol


def _v27_build_block_library(train, agg_cache_full, expected_block, n_blocks, rng):
    """Random historical PR blocks collapsed to duration-conditioned PIT scores.

    The library mixes realised lengths, but each row is PIT-ed against ITS OWN L-day historical law.
    This is what makes cross-duration pooling meaningful without pretending that +x% has the same
    severity over 5 and 40 days.
    """
    n = len(train)
    p = min(max(1.0 / max(float(expected_block), 1.0), 1e-9), 1.0)
    Ls = np.minimum(rng.geometric(p, size=int(n_blocks)), int(V27_BLOCK_MAX_DAYS)).astype(int)
    Ls = np.minimum(Ls, max(n - 2, 1))
    starts = np.array([rng.integers(0, n - L + 1) for L in Ls], dtype=int)
    A = train.to_numpy(dtype=float)
    cs = np.vstack([np.zeros((1, A.shape[1])), np.cumsum(A, axis=0)])
    vals = np.vstack([cs[s + L] - cs[s] for s, L in zip(starts, Ls)])
    end_pos = starts + Ls - 1
    z = np.empty_like(vals)
    # exact-duration ECDFs are read from the precomputed full cache, clipped point-in-time at train end
    end_date = train.index[-1]
    for L in np.unique(Ls):
        m = Ls == L
        hist = agg_cache_full[int(L)].loc[:end_date]
        z[m] = _v27_pit_apply(vals[m], _v27_sorted_marginals(hist), V27_PIT_CLIP)
    return {
        "Z": pd.DataFrame(z, columns=train.columns),
        "length": Ls,
        "end_pos": end_pos,
        "end_date": train.index[end_pos],
    }


def _v27_norm_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
    if w.sum() <= 0:
        w = np.ones_like(w)
    return w / w.sum()


def _v27_fit_hierarchy_custom_weights(Zdf, labels, weights, max_global=3):
    """v24 global/local idea in duration-PIT space, but with arbitrary recency x length weights."""
    D = np.asarray(Zdf, dtype=float)
    w = _v27_norm_weights(weights)
    T, k = D.shape
    labels = np.asarray(labels, dtype=object)
    blocks = list(dict.fromkeys(labels.tolist()))
    mu = (D * w[:, None]).sum(axis=0)
    Dc = D - mu
    scale = np.ones(k)
    for b in blocks:
        m = labels == b
        Cb = (Dc[:, m] * w[:, None]).T @ Dc[:, m]
        lam1 = float(np.linalg.eigvalsh(np.atleast_2d(Cb))[-1])
        scale[m] = 1.0 / np.sqrt(max(lam1, 1e-12))
    Ds = Dc * scale
    S = (Ds * w[:, None]).T @ Ds
    mask_off = labels[:, None] != labels[None, :]
    sd0 = np.sqrt(np.clip(np.diag((Dc * w[:, None]).T @ Dc), 1e-12, None))
    ess = 1.0 / np.sum(w ** 2)
    tol = max(0.05, 3.5 / np.sqrt(max(ess, 1.0)))
    fits = {}
    prev_g = None
    kmax = min(int(max_global), k - 1)
    for kg in range(1, kmax + 1):
        extra = []
        if prev_g is not None:
            prev_W = np.linalg.lstsq(Ds, prev_g, rcond=None)[0]
            extra.append(np.hstack([prev_W, np.eye(k)[:, [0]]]))
        W = _crossblock_factor_weights(S, mask_off, labels, kg, extra_starts=extra)
        g = _nested_unit_factors(Ds, W, w, prev_g)
        lam = (Dc * w[:, None]).T @ g
        U = Dc - g @ lam.T
        C_u = (U * w[:, None]).T @ U
        Roff = np.where(mask_off, np.abs(C_u / np.outer(sd0, sd0)), -1.0)
        rmax = float(Roff[mask_off].max()) if mask_off.any() else 0.0
        fits[kg] = (g, lam, U, rmax)
        prev_g = g
    ok = [kg for kg, (_, _, _, rmax) in fits.items() if rmax <= tol]
    kg = min(ok) if ok else max(fits)
    g, lam, U, rmax = fits[kg]
    groups = [np.where(labels == b)[0] for b in blocks]
    return {"mean": mu, "g": g, "loadings": lam, "u": U, "groups": groups,
            "k_global": kg, "weights": w, "ess": ess, "resid_max": rmax}


def _v27_hier_draws(fit, n_scen, rng):
    """Empirical global/local resampling in PIT-score space; no Gaussian copula is imposed."""
    w = fit["weights"]
    T = len(w)
    gi = rng.choice(T, size=int(n_scen), replace=True, p=w)
    G = fit["g"][gi]
    Ustar = np.zeros((int(n_scen), fit["u"].shape[1]))
    for bi, cols in enumerate(fit["groups"]):
        ui = rng.choice(T, size=int(n_scen), replace=True, p=w)
        Ustar[:, cols] = fit["u"][ui][:, cols]
    return fit["mean"][None, :] + G @ fit["loadings"].T + Ustar


def _v27_energy_score(draws, y, scale):
    """Scaled multivariate energy score; lower is better. Pair term uses disjoint pairs (O(N), reproducible)."""
    X = np.asarray(draws, dtype=float) / np.asarray(scale, dtype=float)[None, :]
    yy = np.asarray(y, dtype=float) / np.asarray(scale, dtype=float)
    t1 = float(np.linalg.norm(X - yy[None, :], axis=1).mean())
    n2 = (len(X) // 2) * 2
    if n2 < 2:
        return t1
    pair = np.linalg.norm(X[:n2:2] - X[1:n2:2], axis=1).mean()
    return float(t1 - 0.5 * pair)

In [3]:
# Minimal data load for the adjudication panel only
stock_prices_raw, _, rates_raw = download_market_data(
    tickers=TICKERS, market_proxy=MARKET_PROXY, rate_tickers=RATE_TICKERS,
    start_date=EQUITY_START, download_end=DOWNLOAD_END)
fx_rates_raw = download_fx_rates(FX_SERIES, FX_START, DOWNLOAD_END)
moodys_raw = download_moodys_credit(MOODYS_SERIES, MOODYS_START_DATE, DOWNLOAD_END)

# Freeze the research sample at the configured evaluation date.
stock_prices_raw = stock_prices_raw.loc[:AS_OF_DATE].dropna(how="all")
rates_raw = rates_raw.loc[:AS_OF_DATE].dropna(how="all")
fx_rates_raw = fx_rates_raw.loc[:AS_OF_DATE].dropna(how="all")
moodys_raw = moodys_raw.loc[:AS_OF_DATE].dropna(how="all")

# Long-history 2-point credit driver used by the compact gate.
_dgs10 = rates_raw["DGS10"].dropna()
credit_driver_raw = pd.DataFrame({
    "Aaa_spr": moodys_raw["Aaa"] - _dgs10,
    "Baa_spr": moodys_raw["Baa"] - _dgs10,
}).dropna(how="any")

v30_panel, v30_labels = _v27_observable_panel()
v30_panel = v30_panel.loc[:AS_OF_DATE].dropna().sort_index()
print(f"v30 compact panel: {v30_panel.index.min().date()} -> {v30_panel.index.max().date()} | "
      f"{len(v30_panel)} rows x {v30_panel.shape[1]} variables")
print(list(v30_panel.columns))


v30 compact panel: 1999-01-05 -> 2026-05-01 | 6810 rows x 7 variables
['EQ_EW', 'RATE_DGS2', 'RATE_DGS10', 'RATE_DGS30', 'CREDIT_Aaa_spr', 'CREDIT_Baa_spr', 'FX_EURUSD']


In [4]:
# v30 scoring / schedule helpers
def _v30_model_name(representation, architecture, propagation):
    return f"{representation}|{architecture}|{propagation}"


def _v30_empirical_draws(frame, n_scen, rng):
    A = np.asarray(frame, dtype=float)
    return A[rng.integers(0, len(A), size=int(n_scen))]


def _v30_pit_blockwise_draws(Zlib, sorted_target, n_scen, rng):
    Z = np.asarray(Zlib, dtype=float)
    ii = rng.integers(0, len(Z), size=int(n_scen))
    return _v27_rank_map_to_duration(Z[ii], sorted_target)


def _v30_variogram_score(draws, y, scale, labels, p=0.5):
    """Scaled variogram score. Returns total + the four cross-block components used for adjudication."""
    X = np.asarray(draws, dtype=float) / np.asarray(scale, dtype=float)[None, :]
    yy = np.asarray(y, dtype=float) / np.asarray(scale, dtype=float)
    labels = np.asarray(labels, dtype=object)
    k = X.shape[1]
    groups = {"EQ_RATES": [], "EQ_CREDIT": [], "RATES_CREDIT": [], "FX_CROSS": []}
    terms = []
    for i in range(k):
        for j in range(i + 1, k):
            obs = abs(yy[i] - yy[j]) ** float(p)
            pred = np.mean(np.abs(X[:, i] - X[:, j]) ** float(p))
            term = float((obs - pred) ** 2)
            terms.append(term)
            a, b = labels[i], labels[j]
            pair = {a, b}
            if pair == {"equity", "rates"}: groups["EQ_RATES"].append(term)
            elif pair == {"equity", "credit"}: groups["EQ_CREDIT"].append(term)
            elif pair == {"rates", "credit"}: groups["RATES_CREDIT"].append(term)
            if (a == "fx") ^ (b == "fx"): groups["FX_CROSS"].append(term)
    total = float(np.mean(terms)) if terms else np.nan
    comp = {g: (float(np.mean(v)) if v else np.nan) for g, v in groups.items()}
    return total, comp


def _v30_stationary_bootstrap_ci(diff, reps=999, seed=30030, level=0.95):
    """Paired stationary-bootstrap CI; positive difference means the augmented model is better."""
    x = np.asarray(diff, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    out = {"n": n, "mean_gain": np.nan, "median_gain": np.nan, "win_rate": np.nan,
           "ci_lo": np.nan, "ci_hi": np.nan, "p_gain_le_0": np.nan, "bootstrap_expected_block": np.nan}
    if n == 0:
        return out
    out.update(mean_gain=float(np.mean(x)), median_gain=float(np.median(x)), win_rate=float(np.mean(x > 0)))
    if n == 1:
        return out
    try:
        br = politis_white_block_length(x[:, None])
        bf = np.asarray(br, dtype=float)
        b = float(np.nanmedian(bf[np.isfinite(bf)])) if np.isfinite(bf).any() else 1.0
    except Exception:
        b = 1.0
    b = float(np.clip(b, 1.0, max(1.0, n / 3.0)))
    pr = min(max(1.0 / b, 1e-9), 1.0)
    rng = np.random.default_rng(int(seed))
    boots = np.empty(int(reps))
    for r in range(int(reps)):
        vals = []
        while len(vals) < n:
            st = int(rng.integers(0, n)); L = int(rng.geometric(pr))
            take = min(L, n - len(vals))
            vals.extend(x[(st + np.arange(take)) % n].tolist())
        boots[r] = np.mean(vals)
    alpha = 1.0 - float(level)
    out.update(ci_lo=float(np.quantile(boots, alpha/2)), ci_hi=float(np.quantile(boots, 1-alpha/2)),
               p_gain_le_0=float((1 + np.sum(boots <= 0)) / (len(boots) + 1)),
               bootstrap_expected_block=b)
    return out


def _v30_outer_schedule(panel):
    """Non-overlapping OOS origins by horizon, evenly thinned only when needed for runtime."""
    rows = []
    n = len(panel)
    for hname, h in HORIZON_BDAYS_BY_MODE.items():
        start = int(V30_MIN_TRAIN_DAYS)
        stop = n - int(h) - 1
        if stop <= start:
            continue
        pos = np.arange(start, stop + 1, int(h), dtype=int)
        cap = V30_MAX_ORIGINS_BY_HORIZON.get(hname)
        if cap is not None and len(pos) > int(cap):
            ii = np.unique(np.round(np.linspace(0, len(pos)-1, int(cap))).astype(int))
            pos = pos[ii]
        rows.extend({"horizon": hname, "asof": panel.index[p], "pos": int(p)} for p in pos)
    return pd.DataFrame(rows).sort_values(["horizon", "asof"]).reset_index(drop=True)


In [5]:
# v30 lean 2x2x2 gate: Energy Score + Variogram Score
print("=== v30 lean adjudication: duration-PIT × architecture × propagation ===")
max_h = max(HORIZON_BDAYS_BY_MODE.values())
agg_cache = _v27_precompute_duration_aggregates(v30_panel, max_h)
outer = _v30_outer_schedule(v30_panel)
print("OOS origins:", outer.groupby("horizon").size().to_dict())

reps = ("raw_duration", "pit_pooled")
archs = ("blockwise", "hierarchical")
props = ("direct_H", "recursive_1W")
score_rows = []

for oi, row in outer.iterrows():
    hname = str(row["horizon"]); h = int(HORIZON_BDAYS_BY_MODE[hname]); pos = int(row["pos"]); asof = pd.Timestamp(row["asof"])
    train = v30_panel.iloc[:pos+1]
    future = v30_panel.iloc[pos+1:pos+1+h]
    if len(future) != h:
        continue
    y = future.sum(axis=0).to_numpy(float)
    aggH = agg_cache[h].loc[:asof]
    if len(aggH) < 100:
        continue
    sortedH = _v27_sorted_marginals(aggH)
    scaleH = aggH.std(ddof=0).replace(0.0, np.nan).fillna(1.0).to_numpy(float)

    bexp, _ = _v27_expected_block_length(train)
    seed0 = int(V30_SEED + 10007*oi + 97*h)
    lib = _v27_build_block_library(train, agg_cache, bexp, V30_BLOCK_LIBRARY_SIZE, np.random.default_rng(seed0+11))
    Zlib = lib["Z"]
    pit_fit = _v27_fit_hierarchy_custom_weights(Zlib, v30_labels, np.ones(len(Zlib)), max_global=V30_HIER_MAX_GLOBAL)

    raw_frames, raw_hfits, sorted_cache = {}, {}, {}
    def raw_frame(L):
        L = int(L)
        if L not in raw_frames: raw_frames[L] = agg_cache[L].loc[:asof]
        return raw_frames[L]
    def sortL(L):
        L = int(L)
        if L not in sorted_cache: sorted_cache[L] = _v27_sorted_marginals(raw_frame(L))
        return sorted_cache[L]
    def raw_hfit(L):
        L = int(L)
        if L not in raw_hfits:
            F = raw_frame(L)
            raw_hfits[L] = _v27_fit_hierarchy_custom_weights(F, v30_labels, np.ones(len(F)), max_global=V30_HIER_MAX_GLOBAL)
        return raw_hfits[L]

    def one_chunk(rep, arch, L, seed):
        L = int(L); rng = np.random.default_rng(int(seed))
        if rep == "raw_duration":
            F = raw_frame(L)
            if arch == "blockwise":
                return _v30_empirical_draws(F, V30_SCENARIOS, rng)
            z = _v27_hier_draws(raw_hfit(L), V30_SCENARIOS, rng)
            return _v27_rank_map_to_duration(z, sortL(L))
        if arch == "blockwise":
            return _v30_pit_blockwise_draws(Zlib, sortL(L), V30_SCENARIOS, rng)
        z = _v27_hier_draws(pit_fit, V30_SCENARIOS, rng)
        return _v27_rank_map_to_duration(z, sortL(L))

    draws = {}
    for ri, rep in enumerate(reps):
        for ai, arch in enumerate(archs):
            base_seed = seed0 + 1000 + 101*ri + 17*ai
            draws[_v30_model_name(rep, arch, "direct_H")] = one_chunk(rep, arch, h, base_seed)
            rem, parts, jj = h, [], 0
            while rem > 0:
                L = min(V30_BASE_HORIZON_DAYS, rem)
                parts.append(one_chunk(rep, arch, L, base_seed + 7919*jj))
                rem -= L; jj += 1
            draws[_v30_model_name(rep, arch, "recursive_1W")] = np.sum(np.stack(parts), axis=0)

    for model, X in draws.items():
        rep, arch, prop = model.split("|")
        es = _v27_energy_score(X, y, scaleH)
        vs, comps = _v30_variogram_score(X, y, scaleH, v30_labels, p=V30_VARIOGRAM_P)
        score_rows.append({"asof":asof,"horizon":hname,"model":model,"representation":rep,"architecture":arch,"propagation":prop,
                           "energy_score":es,"variogram_score":vs, **{f"vs_{k}":v for k,v in comps.items()}})

v30_scores = pd.DataFrame(score_rows)

# Persist point-in-time candidate losses for the objective-aware router.
# These are compact-panel research scores, not tail-risk or downstream-decision scores.
from pathlib import Path as _Path
_v31_root = _Path.cwd()
if _v31_root.name == "notebooks":
    _v31_root = _v31_root.parent
(_v31_root / "results").mkdir(exist_ok=True)
v30_scores.to_csv(_v31_root / "results" / "v31_compact_score_history.csv", index=False)
summary = v30_scores.groupby(["horizon","representation","architecture","propagation"],sort=False).agg(
    n=("energy_score","size"), energy_score=("energy_score","mean"), variogram_score=("variogram_score","mean")).sort_index()
print("\nMean OOS scores (LOWER is better):")
display(summary.round(5))
summary.to_csv("v30_score_summary.csv")

# Headline paired factorial main effects, with stationary-bootstrap CI, for both proper scores.
main_rows = []
for hname, g in v30_scores.groupby("horizon",sort=False):
    for metric in ("energy_score","variogram_score"):
        P = g.pivot(index="asof",columns="model",values=metric).sort_index()
        contrasts = {"PIT_pooling":[],"hierarchy":[],"recursive_1W":[]}
        for a in archs:
            for p in props:
                contrasts["PIT_pooling"].append(P[_v30_model_name("raw_duration",a,p)]-P[_v30_model_name("pit_pooled",a,p)])
        for r in reps:
            for p in props:
                contrasts["hierarchy"].append(P[_v30_model_name(r,"blockwise",p)]-P[_v30_model_name(r,"hierarchical",p)])
        for r in reps:
            for a in archs:
                contrasts["recursive_1W"].append(P[_v30_model_name(r,a,"direct_H")]-P[_v30_model_name(r,a,"recursive_1W")])
        for fi,(factor,arrs) in enumerate(contrasts.items()):
            D = pd.concat(arrs,axis=1).mean(axis=1,skipna=False).dropna().to_numpy()
            stat = _v30_stationary_bootstrap_ci(D,V30_BOOTSTRAP_REPS,V30_SEED+fi+100*HORIZON_BDAYS_BY_MODE[hname])
            main_rows.append({"metric":metric,"factor":factor,"horizon":hname,**stat})
v30_main_effects = pd.DataFrame(main_rows)
print("\nFactorial MAIN effects (POSITIVE = added layer improves the score):")
display(v30_main_effects.set_index(["metric","factor","horizon"])[["n","mean_gain","median_gain","win_rate","ci_lo","ci_hi","p_gain_le_0"]].round(5))
v30_main_effects.to_csv("v30_main_effects.csv",index=False)

# Variogram decomposition: mean paired gains only (no extra bootstrap loop; headline VS above already has CI).
comp_cols = ["vs_EQ_RATES","vs_EQ_CREDIT","vs_RATES_CREDIT","vs_FX_CROSS"]
comp_rows = []
for hname,g in v30_scores.groupby("horizon",sort=False):
    for c in comp_cols:
        P = g.pivot(index="asof",columns="model",values=c).sort_index()
        for factor in ("PIT_pooling","hierarchy","recursive_1W"):
            arrs=[]
            if factor=="PIT_pooling":
                for a in archs:
                    for p in props: arrs.append(P[_v30_model_name("raw_duration",a,p)]-P[_v30_model_name("pit_pooled",a,p)])
            elif factor=="hierarchy":
                for r in reps:
                    for p in props: arrs.append(P[_v30_model_name(r,"blockwise",p)]-P[_v30_model_name(r,"hierarchical",p)])
            else:
                for r in reps:
                    for a in archs: arrs.append(P[_v30_model_name(r,a,"direct_H")]-P[_v30_model_name(r,a,"recursive_1W")])
            D=pd.concat(arrs,axis=1).mean(axis=1,skipna=False).dropna()
            comp_rows.append({"horizon":hname,"factor":factor,"component":c.replace("vs_",""),
                              "mean_gain":float(D.mean()),"win_rate":float((D>0).mean()),"n":len(D)})
v30_vs_components=pd.DataFrame(comp_rows)
print("\nVariogram component mean gains (POSITIVE = improvement):")
display(v30_vs_components.pivot_table(index=["factor","horizon"],columns="component",values="mean_gain").round(5))
v30_vs_components.to_csv("v30_variogram_components.csv",index=False)

# Best specification by horizon under each score.
best_rows=[]
S=summary.reset_index()
for metric in ("energy_score","variogram_score"):
    b=S.sort_values(["horizon",metric]).groupby("horizon",sort=False).head(1).copy()
    b["metric"]=metric; b["score"]=b[metric]
    best_rows.append(b[["metric","horizon","representation","architecture","propagation","n","score"]])
v30_best=pd.concat(best_rows,ignore_index=True)
print("\nBest mean-score specification by horizon:")
display(v30_best.set_index(["metric","horizon"]).round(5))
v30_best.to_csv("v30_best_models.csv",index=False)

# Hard sanity check: at 1W direct-H and recursive-1W must be identical for both scores.
idrows=[]
g=v30_scores[v30_scores.horizon=="1W"]
for metric in ("energy_score","variogram_score"):
    P=g.pivot(index="asof",columns="model",values=metric)
    for r in reps:
        for a in archs:
            d=_v30_model_name(r,a,"direct_H"); q=_v30_model_name(r,a,"recursive_1W")
            idrows.append({"metric":metric,"representation":r,"architecture":a,
                           "max_abs_diff":float(np.nanmax(np.abs(P[d]-P[q])))})
v30_identity=pd.DataFrame(idrows)
print("\n1W direct-vs-recursive identity check (should be zero):")
display(v30_identity.round(12))


=== v30 lean adjudication: duration-PIT × architecture × propagation ===
OOS origins: {'1M': 200, '1W': 200, '1Y': 24, '3M': 80}

Mean OOS scores (LOWER is better):


n  energy_score  \
horizon representation architecture propagation                       
1M      pit_pooled     blockwise    direct_H      200      1.581460   
                                    recursive_1W  200      1.584290   
                       hierarchical direct_H      200      1.582050   
                                    recursive_1W  200      1.592860   
        raw_duration   blockwise    direct_H      200      1.585210   
                                    recursive_1W  200      1.604890   
                       hierarchical direct_H      200      1.584240   
                                    recursive_1W  200      1.587340   
1W      pit_pooled     blockwise    direct_H      200      1.558240   
                                    recursive_1W  200      1.558240   
                       hierarchical direct_H      200      1.553960   
                                    recursive_1W  200      1.553960   
        raw_duration   blockwise    direct_H      200      1.554340   
                                    recursive_1W  200      1.554340   
                       hierarchical direct_H      200      1.553270   
                                    recursive_1W  200      1.553270   
1Y      pit_pooled     blockwise    direct_H       24      2.157210   
                                    recursive_1W   24      2.128950   
                       hierarchical direct_H       24      2.142380   
                                    recursive_1W   24      2.109070   
        raw_duration   blockwise    direct_H       24      2.169540   
                                    recursive_1W   24      2.115710   
                       hierarchical direct_H       24      2.150030   
                                    recursive_1W   24      2.120140   
3M      pit_pooled     blockwise    direct_H       80      1.645480   
                                    recursive_1W   80      1.652820   
                       hierarchical direct_H       80      1.649760   
                                    recursive_1W   80      1.650030   
        raw_duration   blockwise    direct_H       80      1.661090   
                                    recursive_1W   80      1.664590   
                       hierarchical direct_H       80      1.637900   
                                    recursive_1W   80      1.655440   

                                                  variogram_score  
horizon representation architecture propagation                    
1M      pit_pooled     blockwise    direct_H             0.177590  
                                    recursive_1W         0.178440  
                       hierarchical direct_H             0.178280  
                                    recursive_1W         0.179270  
        raw_duration   blockwise    direct_H             0.178870  
                                    recursive_1W         0.178820  
                       hierarchical direct_H             0.178220  
                                    recursive_1W         0.178320  
1W      pit_pooled     blockwise    direct_H             0.169020  
                                    recursive_1W         0.169020  
                       hierarchical direct_H             0.168480  
                                    recursive_1W         0.168480  
        raw_duration   blockwise    direct_H             0.168920  
                                    recursive_1W         0.168920  
                       hierarchical direct_H             0.168640  
                                    recursive_1W         0.168640  
1Y      pit_pooled     blockwise    direct_H             0.230050  
                                    recursive_1W         0.224470  
                       hierarchical direct_H             0.231160  
                                    recursive_1W         0.221290  
        raw_duration   blockwise    direct_H             0.227060  
                                    recursive_1W         0.223060  
     


Factorial MAIN effects (POSITIVE = added layer improves the score):


n  mean_gain  median_gain  win_rate  \
metric          factor       horizon                                          
energy_score    PIT_pooling  1M       200   0.005250     0.005590  0.555000   
                hierarchy    1M       200   0.002340     0.000450  0.520000   
                recursive_1W 1M       200  -0.009100    -0.007710  0.400000   
variogram_score PIT_pooling  1M       200   0.000160    -0.000170  0.485000   
                hierarchy    1M       200  -0.000090     0.000070  0.505000   
                recursive_1W 1M       200  -0.000470    -0.000430  0.475000   
energy_score    PIT_pooling  1W       200  -0.002290    -0.003820  0.465000   
                hierarchy    1W       200   0.002670     0.002270  0.520000   
                recursive_1W 1W       200   0.000000     0.000000  0.000000   
variogram_score PIT_pooling  1W       200   0.000030    -0.000060  0.495000   
                hierarchy    1W       200   0.000420     0.000310  0.545000   
                recursive_1W 1W       200   0.000000     0.000000  0.000000   
energy_score    PIT_pooling  1Y        24   0.004450     0.012640  0.541670   
                hierarchy    1Y        24   0.012450     0.015400  0.583330   
                recursive_1W 1Y        24   0.036320     0.025560  0.583330   
variogram_score PIT_pooling  1Y        24  -0.001830    -0.000190  0.458330   
                hierarchy    1Y        24   0.000670     0.002500  0.583330   
                recursive_1W 1Y        24   0.007600    -0.000220  0.500000   
energy_score    PIT_pooling  3M        80   0.005240     0.007590  0.575000   
                hierarchy    3M        80   0.007710     0.008670  0.575000   
                recursive_1W 3M        80  -0.007160    -0.012490  0.400000   
variogram_score PIT_pooling  3M        80  -0.000590     0.000340  0.525000   
                hierarchy    3M        80   0.000210    -0.000050  0.487500   
                recursive_1W 3M        80  -0.000480    -0.001580  0.412500   

                                         ci_lo     ci_hi  p_gain_le_0  
metric          factor       horizon                                   
energy_score    PIT_pooling  1M       0.000530  0.010030     0.015000  
                hierarchy    1M      -0.002990  0.007510     0.201000  
                recursive_1W 1M      -0.013990 -0.004270     1.000000  
variogram_score PIT_pooling  1M      -0.000830  0.001130     0.356000  
                hierarchy    1M      -0.000900  0.000720     0.581000  
                recursive_1W 1M      -0.002030  0.001390     0.711000  
energy_score    PIT_pooling  1W      -0.007660  0.003330     0.798000  
                hierarchy    1W      -0.003720  0.009580     0.206000  
                recursive_1W 1W       0.000000  0.000000     1.000000  
variogram_score PIT_pooling  1W      -0.001080  0.001180     0.504000  
                hierarchy    1W      -0.000540  0.001430     0.205000  
                recursive_1W 1W       0.000000  0.000000     1.000000  
energy_score    PIT_pooling  1Y      -0.010670  0.020390     0.278000  
                hierarchy    1Y      -0.004720  0.032110     0.106000  
                recursive_1W 1Y      -0.019050  0.090960     0.094000  
variogram_score PIT_pooling  1Y      -0.006370  0.002120     0.800000  
                hierarchy    1Y      -0.002210  0.003110     0.306000  
                recursive_1W 1Y      -0.001880  0.018950     0.056000  
energy_score    PIT_pooling  3M      -0.002400  0.013100     0.088000  
                hierarchy    3M      -0.000280  0.015390     0.029000  
                recursive_1W 3M      -0.018340  0.003570     0.887000  
variogram_score PIT_pooling  3M      -0.002180  0.000950     0.769000  
                hierarchy    3M      -0.001000  0.001420     0.349000  
                recursive_1W 3M      -0.002990  0.002210     0.621000


Variogram component mean gains (POSITIVE = improvement):


component             EQ_CREDIT  EQ_RATES  FX_CROSS  RATES_CREDIT
factor       horizon                                             
PIT_pooling  1M        0.000420  0.001970  0.000210     -0.000450
             1W       -0.000050  0.000500 -0.001090      0.001700
             1Y       -0.016130  0.005610 -0.004190      0.001000
             3M       -0.000600  0.000070 -0.000250     -0.001430
hierarchy    1M        0.001310 -0.001330 -0.000670      0.000670
             1W       -0.002020  0.001690  0.000080      0.001890
             1Y       -0.009500  0.001440  0.000550      0.004440
             3M       -0.000740  0.000090  0.000820      0.000830
recursive_1W 1M       -0.001710 -0.002390  0.000570      0.000520
             1W        0.000000  0.000000  0.000000      0.000000
             1Y        0.019090  0.014170  0.005190      0.005210
             3M        0.000170 -0.003750  0.001340      0.001040


Best mean-score specification by horizon:


representation  architecture   propagation    n  \
metric          horizon                                                   
energy_score    1M          pit_pooled     blockwise      direct_H  200   
                1W        raw_duration  hierarchical      direct_H  200   
                1Y          pit_pooled  hierarchical  recursive_1W   24   
                3M        raw_duration  hierarchical      direct_H   80   
variogram_score 1M          pit_pooled     blockwise      direct_H  200   
                1W          pit_pooled  hierarchical      direct_H  200   
                1Y        raw_duration  hierarchical  recursive_1W   24   
                3M        raw_duration  hierarchical      direct_H   80   

                           score  
metric          horizon           
energy_score    1M      1.581460  
                1W      1.553270  
                1Y      2.109070  
                3M      1.637900  
variogram_score 1M      0.177590  
                1W      0.168480  
                1Y      0.219290  
                3M      0.206610


1W direct-vs-recursive identity check (should be zero):


,metric,representation,architecture,max_abs_diff
0,energy_score,raw_duration,blockwise,0.000000
1,energy_score,raw_duration,hierarchical,0.000000
2,energy_score,pit_pooled,blockwise,0.000000
3,energy_score,pit_pooled,hierarchical,0.000000
4,variogram_score,raw_duration,blockwise,0.000000
5,variogram_score,raw_duration,hierarchical,0.000000
6,variogram_score,pit_pooled,blockwise,0.000000
7,variogram_score,pit_pooled,hierarchical,0.000000
